## Forecasting Simulations

can I use the simulated frames and the nodesnapshot baseline with xgboost?

In [4]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import duckdb as ddb

In [5]:
con = ddb.connect(":memory:")

In [6]:
def load_experiment(name, path):
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT *
        FROM read_parquet('{RESULTS_DIR / path["path"]}')
    """)

In [7]:
ROOT = Path.cwd().resolve().parents[1]
RESULTS_DIR = ROOT / "capacity_based_results/"
DATA_DIR = ROOT / "datasets/cloud_energy_consumption/processed"

# Load Baselines
con.execute(f"""CREATE OR REPLACE VIEW baseline AS SELECT * FROM read_parquet('{DATA_DIR}/node_snapshot.parquet')""")

EXPERIMENTS = {
    "mm_pbfd_0_30": {
    "path": "MM_PBFD_0_30/simulated_MM_PBFD_0_30.parquet",
    "label": "MM/POWER (0-30)",
    "group": "MM",
    "type": "POWER"
    },
}

for view_name, cfg in EXPERIMENTS.items():
    load_experiment(view_name, cfg)

In [12]:
simulation = con.execute("SELECT * FROM mm_pbfd_0_30").df()
simulation.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,timestamp,node_name,node_group,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power,vm_ids,vm_hypervisor_groups,vm_cpus,vm_memories_mb,vm_powers
0,2024-12-14 01:00:00+01:00,00366801,a6177608,0.23,404.17,128,1100.0,386653.664062,366137.002604,0,0.0,0.0,0.0,404.17,0.0,[],[],[],[],[]
1,2024-12-14 01:00:00+01:00,0049db0c,a6177608,0.21,415.00,128,1100.0,386653.667969,289728.684245,0,0.0,0.0,0.0,415.00,0.0,[],[],[],[],[]
2,2024-12-14 01:00:00+01:00,0064a367,a6177608,99.88,784.17,128,1100.0,386653.667969,1589.156901,0,0.0,0.0,0.0,784.17,0.0,[],[],[],[],[]
3,2024-12-14 01:00:00+01:00,0065ef1b,a6177608,0.03,410.00,128,1100.0,386653.664062,366362.100911,0,0.0,0.0,0.0,410.00,0.0,[],[],[],[],[]
4,2024-12-14 01:00:00+01:00,05c5ef00,a6177608,18.52,465.00,128,1100.0,386653.667969,1432.229167,0,0.0,0.0,0.0,465.00,0.0,[],[],[],[],[]


In [13]:
simulation.columns

Index(['timestamp', 'node_name', 'node_group', 'cpu_usage_percent',
       'ipmi_system_power_watts', 'cpu_capacity', 'power_capacity',
       'memory_capacity_mb', 'memory_available_mb', 'vm_count',
       'vm_power_allocated', 'cpu_allocated', 'memory_allocated_mb',
       'baseline_power', 'simulated_power', 'vm_ids', 'vm_hypervisor_groups',
       'vm_cpus', 'vm_memories_mb', 'vm_powers'],
      dtype='str')

In [14]:
baseline = con.execute("SELECT * FROM baseline").df()
baseline.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,timestamp,node_name,node_group,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power
0,2024-12-14 01:00:00+01:00,00366801,a6177608,0.23,404.17,128,1100.0,386653.664062,366137.002604,0,0.0,0.0,0.0,404.17,404.17
1,2024-12-14 01:00:00+01:00,0049db0c,a6177608,0.21,415.00,128,1100.0,386653.667969,289728.684245,0,0.0,0.0,0.0,415.00,415.00
2,2024-12-14 01:00:00+01:00,0064a367,a6177608,99.88,784.17,128,1100.0,386653.667969,1589.156901,0,0.0,0.0,0.0,784.17,784.17
3,2024-12-14 01:00:00+01:00,0065ef1b,a6177608,0.03,410.00,128,1100.0,386653.664062,366362.100911,0,0.0,0.0,0.0,410.00,410.00
4,2024-12-14 01:00:00+01:00,05c5ef00,a6177608,18.52,465.00,128,1100.0,386653.667969,1432.229167,0,0.0,0.0,0.0,465.00,465.00


In [15]:
baseline.columns

Index(['timestamp', 'node_name', 'node_group', 'cpu_usage_percent',
       'ipmi_system_power_watts', 'cpu_capacity', 'power_capacity',
       'memory_capacity_mb', 'memory_available_mb', 'vm_count',
       'vm_power_allocated', 'cpu_allocated', 'memory_allocated_mb',
       'baseline_power', 'simulated_power'],
      dtype='str')